In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision.transforms import transforms
import torchtext


In [3]:
torch.manual_seed(1234)
torch.cuda.manual_seed(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [20]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [21]:
train_data = torchvision.datasets.CIFAR10(root='./data1', train=True, download=True, transform=transform)
test_data = torchvision.datasets.CIFAR10(root='./data1', train=False, download=True, transform=transform)

train_size = int(0.8 * len(train_data))
valid_size = len(train_data) - train_size
train_data, valid_data = random_split(train_data, [train_size, valid_size])

Files already downloaded and verified
Files already downloaded and verified


In [22]:
test_data[0][0].shape, test_data[0][1]

(torch.Size([3, 32, 32]), 3)

In [23]:
# test_data[0]

In [24]:
train_data.dataset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data1
    Split: Train
    StandardTransform
Transform: Compose(
               RandomHorizontalFlip(p=0.5)
               RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [25]:
# train_data[0]

In [26]:
BATCH_SIZE = 64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [28]:
# dim = (size - k + 2 * padding) / stride + 1
# (32 - 3 + 2) / 2 + 1 = 15.5 + 1 = 15 + 1 = 16
# (16 - 3 + 2) / 2 + 1 = 7.5 + 1 = 7 + 1 = 8
# (8 - 3 + 2) / 2 + 1 = 3.5 + 1 = 3 + 1 = 4

In [36]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(-1, 128 * 4 * 4)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [37]:
model = SimpleCNN()
model

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2048, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [38]:
for p in model.parameters():
  print(type(p), p.size())
  # print(p.numel())

<class 'torch.nn.parameter.Parameter'> torch.Size([32, 3, 3, 3])
<class 'torch.nn.parameter.Parameter'> torch.Size([32])
<class 'torch.nn.parameter.Parameter'> torch.Size([64, 32, 3, 3])
<class 'torch.nn.parameter.Parameter'> torch.Size([64])
<class 'torch.nn.parameter.Parameter'> torch.Size([128, 64, 3, 3])
<class 'torch.nn.parameter.Parameter'> torch.Size([128])
<class 'torch.nn.parameter.Parameter'> torch.Size([512, 2048])
<class 'torch.nn.parameter.Parameter'> torch.Size([512])
<class 'torch.nn.parameter.Parameter'> torch.Size([10, 512])
<class 'torch.nn.parameter.Parameter'> torch.Size([10])


In [39]:
# help(model)

In [40]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

criterion.to(device)
model.to(device)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2048, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [41]:
N_EPOCH = 2

for epoch in range(N_EPOCH):
  model.train()
  train_loss = 0
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    out = model(images)
    loss = criterion(out, labels)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()

  val_loss = 0
  model.eval()
  with torch.no_grad():
    for images, labels in valid_loader:
      images, labels = images.to(device), labels.to(device)
      out = model(images)
      loss = criterion(out, labels)
      val_loss += loss.item()

  print(f"Epoch: {epoch}, Train Loss: {train_loss/len(train_loader)}, Valid Loss: {val_loss/len(valid_loader)}")

Epoch: 0, Train Loss: 1.5413487260818481, Valid Loss: 1.2636325587132933
Epoch: 1, Train Loss: 1.175507803916931, Valid Loss: 1.0535996563874992


In [42]:
model.eval()
test_loss = 0
tot = 0
corr = 0
with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    test_loss += loss
    _, pred = torch.max(outputs, 1)
    # print(pred)
    tot += len(labels)
    corr += (pred == labels).sum().item()
print(f"Test Loss: {test_loss/len(test_loader)}, Test Accuracy: {100 * corr / tot}")


Test Loss: 1.0434162616729736, Test Accuracy: 62.94
